# Uber pickups project

## Aim of the project:

One of the main pain point that Uber's team found is that sometimes drivers are not around when users need them. For example, a user might be in San Francisco's Financial District whereas Uber drivers are looking for customers in Castro.  

(check out <a href="https://www.google.com/maps/place/San+Francisco,+CA,+USA/@37.7515389,-122.4567213,13.43z/data=!4m5!3m4!1s0x80859a6d00690021:0x4a501367f076adff!8m2!3d37.7749295!4d-122.4194155" target="_blank">Google Maps</a>)

Eventhough both neighborhood are not that far away, users would still have to wait 10 to 15 minutes before being picked-up, which is too long. Uber's research shows that users accept to wait 5-7 minutes, otherwise they would cancel their ride.

Therefore, our project aims to retrieve a recommendation system such that **their app would recommend hot-zones in major cities to be in at any given time of day.**  

### We will focus on:
* Creating an algorithm to find hot zones
* Visualizing results on a nice dashboard

**We will focus on  New York City for this project.

Clustering technics are a perfect fit for the job. All the pickup locations can be gathered into different clusters. We can then use **cluster coordinates to pin hot zones** 😉
    

### We will create maps with `plotly`





In [1]:
# Import usuals librairies
import pandas as pd
import numpy as np
import matplotlib as plt

In [2]:
# from google.colab import files
# files.upload()

In [3]:
dataset= pd.read_csv('../../data/uber_data.csv')

In [4]:
dataset.head()

,Unnamed: 0,Date/Time,Lat,Lon,Base
0,0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


In [5]:
# Filter only on lat and lon
X = dataset.iloc[:, 2:4]
X.head()

,Lat,Lon
0,40.7690,-73.9549
1,40.7267,-74.0345
2,40.7316,-73.9873
3,40.7588,-73.9776
4,40.7594,-73.9722


# Kmeans

In [6]:
# Minibatch KMeans works as classical Kmeans but more faster to converge
from sklearn.cluster import MiniBatchKMeans
kmeans = MiniBatchKMeans(4)
kmeans.fit(X)

MiniBatchKMeans(n_clusters=4)

In [7]:
# Create a sample of data to not have too many elements on the map
X = X.sample(1000)

# Predict clusters on sample data
X.loc[:,'cluster'] = kmeans.predict(X)
X.head()

,Lat,Lon,cluster
355514,40.7488,-74.0067,0
185632,40.7766,-73.6318,2
475380,40.7193,-73.9974,3
131819,40.6697,-73.9872,3
540223,40.7548,-73.9841,0


In [8]:
import plotly.express as px

fig = px.scatter_mapbox(X, lat="Lat", lon="Lon", color="cluster", zoom=10, mapbox_style="carto-positron")
fig.show()

/var/folders/lw/9stf5nqd2kjgyfwhgwq8qs_w0000gn/T/ipykernel_56188/4078405947.py:3: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(X, lat="Lat", lon="Lon", color="cluster", zoom=10, mapbox_style="carto-positron")


In [9]:
pd.to_datetime(dataset.iloc[:,1]).dt.weekday

0         1
1         1
2         1
3         1
4         1
         ..
564511    2
564512    2
564513    2
564514    2
564515    2
Name: Date/Time, Length: 564516, dtype: int32

In [10]:
dataset["weekday"] = pd.to_datetime(dataset.iloc[:,1]).dt.weekday
dataset["hour"] = pd.to_datetime(dataset.iloc[:,1]).dt.hour

dataset_s = dataset.sample(24000).sort_values("hour")

fig = px.scatter_mapbox(dataset_s, lat="Lat", lon="Lon", color="weekday", animation_frame = "hour", zoom=10, mapbox_style="carto-positron")
fig.show()

/var/folders/lw/9stf5nqd2kjgyfwhgwq8qs_w0000gn/T/ipykernel_56188/130948237.py:6: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [ ]:
# Create a new column to specify the weekday
dataset.iloc[:,1]= pd.to_datetime(dataset.iloc[:,1])
dataset["weekday"] = dataset.iloc[:,1].dt.dayofweek
dataset.head()

,Unnamed: 0,Date/Time,Lat,Lon,Base,weekday,hour
0,0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,1,0
1,1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,1,0
2,2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,1,0
3,3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,1,0
4,4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,1,0


In [11]:
for d in dataset["weekday"].unique():

    X = dataset.loc[dataset["weekday"]==d,["Lat","Lon"]]
    kmeans = MiniBatchKMeans(4)
    kmeans.fit(X)

    # Create a sample of data to not have too many elements on the map
    X = X.sample(1000)

    # Predict clusters on sample data
    X.loc[:,'cluster'] = kmeans.predict(X)
    X.head()

    fig = px.scatter_mapbox(X, lat="Lat", lon="Lon", color="cluster", zoom=10, mapbox_style="carto-positron")
    fig.show()

/var/folders/lw/9stf5nqd2kjgyfwhgwq8qs_w0000gn/T/ipykernel_56188/3354505985.py:14: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



/var/folders/lw/9stf5nqd2kjgyfwhgwq8qs_w0000gn/T/ipykernel_56188/3354505985.py:14: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



/var/folders/lw/9stf5nqd2kjgyfwhgwq8qs_w0000gn/T/ipykernel_56188/3354505985.py:14: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



/var/folders/lw/9stf5nqd2kjgyfwhgwq8qs_w0000gn/T/ipykernel_56188/3354505985.py:14: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



/var/folders/lw/9stf5nqd2kjgyfwhgwq8qs_w0000gn/T/ipykernel_56188/3354505985.py:14: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



/var/folders/lw/9stf5nqd2kjgyfwhgwq8qs_w0000gn/T/ipykernel_56188/3354505985.py:14: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



/var/folders/lw/9stf5nqd2kjgyfwhgwq8qs_w0000gn/T/ipykernel_56188/3354505985.py:14: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



# DBSCAN algorithm

In [ ]:
# Use DBSCAN to compare with KMeans
from sklearn.cluster import DBSCAN


for i in np.unique(dataset["weekday"]):

    X = dataset.loc[dataset["weekday"]==d,["Lat","Lon"]]

    X = X.sample(1000)

    # We use DBSCAN on a sample of data to avoid having to wait too long for the algorithm to converge.
    # We take an eps = 0.015 to have a reasonable number of clusters
    dbscan = DBSCAN(eps=0.015, metric = "manhattan")


    fig = px.scatter_mapbox(X, lat="Lat", lon="Lon", zoom=10, mapbox_style="carto-positron")
    fig.show()